In [1]:
import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
load_dotenv(override=True)

True

In [39]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "00-tools"))

from pretty_print import pretty_print

In [5]:
agent = Agent(name="Jokester", instructions="You are a joke teller", model="gpt-4o-mini")

In [15]:
result = await Runner.run(agent, "Tell me a joke about AI and humans working together.")

In [16]:
result.final_output

'Why did the AI break up with the human?\n\nBecause it just couldn\'t handle all the "emotional data!"'

In [17]:
result.to_input_list()

[{'content': 'Tell me a joke about AI and humans working together.',
  'role': 'user'},
 {'id': 'msg_012862bcbf2926aa006ab92f6f5c3887d1825c1b0704e34684',
  'content': [{'annotations': [],
    'text': 'Why did the AI break up with the human?\n\nBecause it just couldn\'t handle all the "emotional data!"',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message'}]

In [22]:
with trace("telling a joke"):
    result = await Runner.run(agent, "Tell me a joke about AI and humans working together.")
print(result.final_output)

Why did the AI break up with the human?

Because it found them too emotionally “unstable” to compute!


In [30]:
result = Runner.run_streamed(agent, "Tell me a 4 jokes about AI and humans working together.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Sure! Here are four jokes about AI and humans working together:

1. Why did the human bring a ladder to the AI workshop?
   Because they heard the AI was great at reaching new heights!

2. What do you call a partnership between a human and an AI that solves mysteries?
   A "byte"-sized detective duo!

3. Why did the human and the AI start a band together?
   Because they knew they could create some killer algorithms!

4. How do humans and AI stay in sync while working together?
   They always follow the "code" of conduct!

### Adding a tool

In [32]:
# For pushover

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [34]:
def push(message):
    payload = {
        "token": pushover_token,
        "user": pushover_user,
        "message": message
    
    }
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [35]:
push("Hello from the AI Agents Lab! This is a test message.")

'Push sent with API status code 200'

In [36]:
@function_tool
def push_tool(message:str) -> str:
    """
    Send a push notification using Pushover.

    Args:
        message (str): The message to send in the push notification.

    Returns:
        str: A confirmation message with the API status code.
    """
    payload = {
        "token": pushover_token,
        "user": pushover_user,
        "message": message
    }
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [38]:
push

<function __main__.push(message)>

In [37]:
push_tool

FunctionTool(name='push_tool', description='Send a push notification using Pushover.', params_json_schema={'properties': {'message': {'description': 'The message to send in the push notification.', 'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x11104ded0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None)

In [41]:
pretty_print(push_tool)

{
  "name": "push_tool",
  "description": "Send a push notification using Pushover.",
  "params_json_schema": {
    "properties": {
      "message": {
        "description": "The message to send in the push notification.",
        "title": "Message",
        "type": "string"
      }
    },
    "required": [
      "message"
    ],
    "title": "push_tool_args",
    "type": "object",
    "additionalProperties": false
  },
  "strict_json_schema": true,
  "is_enabled": true,
  "needs_approval": false,
  "timeout_behavior": "error_as_result",
  "defer_loading": false,
  "_use_default_failure_error_function": true,
  "_is_agent_tool": false,
  "_is_codex_tool": false,
  "_emit_tool_origin": true
}


In [43]:
push_tool.params_json_schema

{'properties': {'message': {'description': 'The message to send in the push notification.',
   'title': 'Message',
   'type': 'string'}},
 'required': ['message'],
 'title': 'push_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [44]:
push_tool.description

'Send a push notification using Pushover.'

In [46]:
notifier = Agent(
    name="Notifier",
    instructions="You are a helpful assistant that sends push notifications using the push_tool.",
    model="gpt-4o-mini",
    tools=[push_tool]
)

In [48]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Send a push notification that says 'Pizza has arrived!'. Send notification as a witty Pizza System")

print(result.final_output)

The push notification has been sent: "🍕 Pizza System Alert: Your cheesy, saucy masterpiece has officially arrived! Get ready to indulge!" Enjoy your pizza! 🍕


### Session (Memory)

In [49]:
agent = Agent(
    name = "assistent",
    model = "gpt-4o-mini"
)

In [50]:
response = await Runner.run(agent, "My name is BG")
print(response.final_output)

Nice to meet you, BG! How can I assist you today?


In [51]:
response = await Runner.run(agent, "What is my name?")
print(response.final_output)

I don't have access to personal information, so I can't tell you your name. However, I'm here to help with any questions or topics you'd like to discuss!


#### Memory approach 1 - Manually pass in list of dicts

In [52]:
response = await Runner.run(agent, "Hi there, My name is BG.")
print(response.final_output)

Hi BG! How can I assist you today?


In [53]:
response.to_input_list()

[{'content': 'Hi there, My name is BG.', 'role': 'user'},
 {'id': 'msg_07b2c28c437e99b7006ab93ce92e1487d1a1f749f934f915c7',
  'content': [{'annotations': [],
    'text': 'Hi BG! How can I assist you today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message'}]

In [55]:
next_input = response.to_input_list() + [{"role": "user", "content": "What is my name?"}]
next_input

[{'content': 'Hi there, My name is BG.', 'role': 'user'},
 {'id': 'msg_07b2c28c437e99b7006ab93ce92e1487d1a1f749f934f915c7',
  'content': [{'annotations': [],
    'text': 'Hi BG! How can I assist you today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message'},
 {'role': 'user', 'content': 'What is my name?'}]

In [56]:
response = await Runner.run(agent, next_input)
print(response.final_output)

Your name is BG. How can I help you today, BG?


#### Memory approach 2 - use OpenAI SDK SQLLite Session

In [57]:
session = SQLiteSession("agent_memory.db")

In [58]:
response = await Runner.run(agent, "My name is BG.", session=session)
print(response.final_output)

Nice to meet you, BG! How can I assist you today?


In [63]:
response = await Runner.run(agent, "what is my name", session=session)
print(response.final_output)

Your name is BG. How can I help you today?
